In [ ]:
import json
import os
import time
import uuid
from functools import partial
from pathlib import Path
from urllib.parse import urlsplit

import polars as pl
from dotenv import load_dotenv

# ReAct vs. native search

One AnchorBank query, shared BM25 budget. ReAct uses Inspect tools; native uses structured search/stop decisions. Both retrieve context, not answers.

Install from the repository root: `pip install -e ".[inspect,serve]" huggingface_hub`.
The model must support tool calls (ReAct) and structured JSON (native). Hosted API calls are billed by the provider.

In [ ]:
from inspect_ai import eval_async
from inspect_ai.log import read_eval_log
from inspect_ai.model import get_model

from justatom.agentic.benchmark import SearchBudget, SearchSession, score_searches
from justatom.agentic.benchmark_data import load_cases
from justatom.agentic.inspect_harness import build_task
from justatom.agentic.inspect_results import export_results
from justatom.agentic.openai_compatible import OpenAICompatibleChatBackend
from justatom.agentic.schemas import AgentObjective
from justatom.etc.errors import DocumentStoreError
from justatom.retrieval import RetrievalRuntime
from justatom.storing.weaviate import WeaviateDocumentStore

In [ ]:
root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
load_dotenv(root / ".env")

model = dict(
    name="gpt-4.1-mini",
    base_url="https://api.openai.com/v1",
    api_key=os.getenv("OPENAI_API_KEY"),
)
# model = dict(name="search-agent", base_url="http://127.0.0.1:18081/v1", api_key="unused")

store_config = dict(collection="Agentic_tutorial_bm25", url="http://localhost:2211", grpc_port=50051)
overwrite = True
batch_size = 256
output_dir = root / ".data" / "agentic-tutorial"
query_id = "branch-002a0cef92afdc4085a407af"
budget = SearchBudget(top_k=3, max_searches=4, max_context_documents=12,
                      max_document_chars=2000, max_context_chars=12000)
task_options = dict(max_model_calls=3, max_output_tokens=4096, token_limit=24000, time_limit=180)

## Data

The dataset is private: authenticate with `hf auth login` using an account with access, or set `HF_TOKEN`. For GPT, set `OPENAI_API_KEY` in the environment or `.env`.

In [ ]:
from huggingface_hub import snapshot_download

package = Path(snapshot_download(
    "justatom/anchorbank-multihop-retrieval", repo_type="dataset", revision="v1.0.0",
    allow_patterns=["benchmark/validation.parquet", "corpus/corpus.parquet"],
    local_dir=output_dir / "anchorbank",
))
corpus_path = package / "corpus" / "corpus.parquet"
benchmark_path = package / "benchmark" / "validation.parquet"
case = next(case for case in load_cases(benchmark_path) if case.query_id == query_id)
print(case.query)

## BM25 index

A collection is a named set of documents and its search index. This example uses the separate `Agentic_tutorial_bm25` collection.

`overwrite=True` replaces that collection's contents; `False` reuses a nonempty collection. No embeddings are computed. Start Docker, then run `docker compose up -d weaviate` from the repository root.

`RetrievalRuntime(mode="keyword")` owns the store, an `Indexer` and a `KeywordRetriever`. Indexing and both search methods below use that same runtime.

In [ ]:
def validate_endpoint(url):
    parsed = urlsplit(url)
    if (parsed.scheme not in {"http", "https"} or not parsed.hostname
            or parsed.username is not None or parsed.password is not None
            or parsed.query or parsed.fragment):
        raise ValueError("Use an HTTP(S) endpoint without embedded credentials, query or fragment.")

In [ ]:
corpus = pl.read_parquet(corpus_path)
if not {"chunk_id", "content"}.issubset(corpus.columns):
    raise ValueError("Corpus must have chunk_id and content columns")
documents = corpus.select("chunk_id", "content").rename({"chunk_id": "id"}).to_dicts()
if not documents or any(not isinstance(row[key], str) or not row[key].strip()
                        for row in documents for key in ("id", "content")):
    raise ValueError("Corpus must contain nonempty chunk_id and content strings")
if len({row["id"] for row in documents}) != len(documents):
    raise ValueError("Corpus chunk_id values must be unique")

## Search

Gold IDs are scored offline, never sent to the model. `top_k` applies per search; both agents start with the original query.

`build_task` selects Inspect ReAct or `AgenticRAGRuntime`. Native uses `OpenAICompatibleChatBackend`; its Inspect model is only an unused placeholder. The runtime context closes the store even if a run fails.

In [ ]:
validate_endpoint(store_config["url"])
validate_endpoint(model["base_url"])
if type(batch_size) is not int or batch_size <= 0:
    raise ValueError("batch_size must be a positive integer")
score_searches([], [], case.chunk_ids)

run_dir = output_dir / uuid.uuid4().hex[:12]
run_dir.mkdir(parents=True, exist_ok=False)
try:
    store = await WeaviateDocumentStore.connect(**store_config)
    if overwrite:
        try:
            await store.delete_collection()
        finally:
            await store.close()
        store = await WeaviateDocumentStore.connect(**store_config)
except DocumentStoreError as error:
    raise RuntimeError(
        f'Check Weaviate at {store_config["url"]} and gRPC port {store_config.get("grpc_port", 50051)}. '
        "Start Docker, then run 'docker compose up -d weaviate' from the repository root."
    ) from error

async with RetrievalRuntime(store=store, embedder=None, mode="keyword") as retrieval:
    if overwrite or not await store.count_documents():
        await retrieval.index(documents, batch_size=batch_size)

    # Single-search BM25 baseline.
    session = SearchSession(retrieval, budget)
    started = time.perf_counter()
    await session.search(case.query)
    baseline = {
        "query_id": case.query_id, "query": case.query,
        "method": "bm25", "status": "success", "error": None,
        "termination_reason": "single_search", "model_calls": 0,
        "duration_ms": (time.perf_counter() - started) * 1000, "calls": session.calls,
        "context_ids": session.context_ids,
        "context_documents": session.context_documents,
        "evaluation": score_searches(session.calls, session.context_ids, case.chunk_ids),
        "usage": {"total_tokens": 0, "cost_usd": 0},
    }
    (run_dir / "bm25.json").write_text(json.dumps(baseline, ensure_ascii=False) + "\n")

    react_task = build_task([case], retrieval, method="react", budget=budget, **task_options)
    native_task = build_task(
        [case], retrieval, method="native", budget=budget, **task_options,
        planner_factory=partial(
            OpenAICompatibleChatBackend, model["base_url"], model["name"],
            api_key=model.get("api_key") or "unused", objective=AgentObjective.CONTEXT,
            temperature=0, max_tokens=task_options["max_output_tokens"],
            timeout_seconds=task_options["time_limit"],
        ),
    )
    provider = "openrouter" if urlsplit(model["base_url"]).hostname == "openrouter.ai" else "openai-api/justatom"
    logs = []
    try:
        # ReAct: the model calls Inspect's search tool.
        async with get_model(
            f'{provider}/{model["name"]}', base_url=model["base_url"],
            api_key=model.get("api_key") or "unused", max_retries=0,
            responses_api=False, strict_tools=False, memoize=False,
        ) as react_model:
            logs.extend(await eval_async(
                react_task, model=react_model, max_samples=1,
                log_dir=str(run_dir / "logs" / "react"), log_model_api=True,
            ))

        # Native: AgenticRAGRuntime calls the planner directly.
        async with get_model("mockllm/unused", memoize=False) as native_model:
            logs.extend(await eval_async(
                native_task, model=native_model, max_samples=1,
                log_dir=str(run_dir / "logs" / "native"), log_model_api=True,
            ))
    finally:
        export_results(logs, run_dir, cases=[case], methods=["react", "native"])

react, native = [
    {**json.loads(line), "output_dir": run_dir}
    for line in (run_dir / "results.jsonl").read_text().splitlines()
]

## Comparison

In [ ]:
def comparison_row(row):
    quality = row["evaluation"]["final_context"]
    usage = row.get("usage") or {}
    duration = row.get("duration_ms")
    return {
        "method": row["method"], "status": row["status"],
        "hits": quality["hits"], "gold": quality["gold_count"],
        "recall": round(quality["recall"], 3), "all_gold": quality["all_gold"],
        "first_complete_hop": row["evaluation"]["first_complete_hop"],
        "searches": len(row["calls"]), "model_calls": row.get("model_calls"),
        "seconds": None if duration is None else round(duration / 1000, 2),
        "tokens": usage.get("total_tokens"),
        "provider_cost_usd": usage.get("cost_usd"),
        "stop": row.get("termination_reason"),
    }

In [ ]:
results = [baseline, react, native]
response_result = pl.DataFrame([comparison_row(result) for result in results])
response_result.head(10)

## Traces

In [ ]:
def search_steps(result):
    return [
        {
            "hop": call["index"], "query": call["query"],
            "chunk_ids": [doc["chunk_id"] for doc in call["documents"]],
            "hits": coverage["hits"], "gold": coverage["gold_count"],
        }
        for call, coverage in zip(result["calls"], result["evaluation"]["cumulative"])
    ]


def agent_decisions(result):
    if result["method"] == "native":
        return [
            {"action": step["decision"]["action"],
             "query": step["decision"].get("query_text"),
             "reason": step["decision"].get("reason_text")}
            for step in (result.get("native_trace") or {}).get("steps", []) if step.get("decision")
        ]
    return [
        {"tool": call.function, "arguments": call.arguments}
        for path in sorted((Path(result["output_dir"]) / "logs" / result["method"]).glob("*.eval"))
        for sample in read_eval_log(str(path)).samples or []
        for message in sample.messages
        for call in getattr(message, "tool_calls", None) or []
    ]

In [ ]:
react_steps = pl.DataFrame(search_steps(react))
react_steps.head(10)

In [ ]:
react_decisions = pl.DataFrame(agent_decisions(react))
react_decisions.head(10)

In [ ]:
native_steps = pl.DataFrame(search_steps(native))
native_steps.head(10)

In [ ]:
native_decisions = pl.DataFrame(agent_decisions(native))
native_decisions.head(10)

## Retained context

In [ ]:
context_result = pl.DataFrame(native["context_documents"])
context_result.head(10)

ID coverage is not answer correctness. This is a smoke example, not a representative benchmark; timing depends on cache state and run order.

Logs stay under `result["output_dir"]`; open them with `inspect view --log-dir <output_dir>/logs`.